# Benchmark de Modelos Grandes (Transformers) para Detecção de Phishing

**Dataset:** `kmack/Phishing_urls` (HuggingFace) — 708k URLs  
**Entrada:** URL bruta + enriquecimento WHOIS (`domain_age_days`, `registrar`, `status`)  
**Hardware:** Google Colab T4 (16 GB VRAM) com `fp16` habilitado  

**Modelos avaliados:**
- DomURLs-BERT (`amahdaouy/DomURLs_BERT`) — ~110M params, pré-treinado em URLs/domínios maliciosos (cybersec)
- CANINE-s (`google/canine-s`) — ~121M params, transformer de nível de caractere (sem tokenização por subpalavras)
- XLNet-base (`xlnet-base-cased`) — ~117M params, permutation language modeling (autoregressive)
- SecureBERT (`ehsanaghaei/SecureBERT`) — ~125M params, RoBERTa pré-treinado em corpus de cibersegurança

**Diversidade arquitetural:**
- DomURLs-BERT e SecureBERT: BERT/RoBERTa com domínio cybersec (pré-treino especializado)
- CANINE-s: opera em caracteres brutos — sem vocabulário fixo, ideal para URLs
- XLNet: atenção permutada autoregressive — fundamentalmente diferente do masked LM do BERT

**Cache automático no Google Drive:**  
Cada modelo verifica se já foi treinado. Se o checkpoint existir no Drive, o treinamento é pulado e o modelo é carregado diretamente.

**Pré-requisito:** fazer upload do `whois_cache.json` para `/content/` no Colab antes de rodar.

In [ ]:
# ============================================================
# 1. Verificação de GPU e Instalação
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "AVISO: GPU não detectada!")

!pip install -q transformers datasets accelerate scikit-learn tldextract
print("Instalação concluída!")

In [ ]:
# ============================================================
# 2. Imports
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time, os, gc, json, warnings
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    CanineForSequenceClassification,
    Trainer, TrainingArguments,
)
from datasets import load_dataset
import tldextract

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc,
    confusion_matrix, ConfusionMatrixDisplay,
    log_loss, matthews_corrcoef,
    precision_recall_curve, average_precision_score,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB")
else:
    print("AVISO: Sem GPU. Ative em Runtime → Alterar tipo de runtime → T4 GPU")

torch.manual_seed(42)
np.random.seed(42)
print("Imports OK.")

In [ ]:
# ============================================================
# 3. Configuração Global
# ============================================================
RANDOM_STATE      = 42
MAX_TRAIN_SAMPLES = 200_000   # None = dataset completo (~566k)
MAX_EVAL_SAMPLES  = 50_000    # None = avaliação completa (~142k)
MAX_LENGTH        = 128
NUM_EPOCHS        = 3
LR                = 2e-5
WARMUP_RATIO      = 0.06
WEIGHT_DECAY      = 0.01

# --- Google Drive ---
DRIVE_BASE = "/content/drive/MyDrive/TCC-Phishing-Benchmark"

# --- WHOIS (faça upload manual para /content/ antes de rodar) ---
WHOIS_PATH = "/content/whois_cache.json"
if not os.path.exists(WHOIS_PATH):
    WHOIS_PATH = f"{DRIVE_BASE}/whois_cache.json"  # fallback no Drive

print(f"DRIVE_BASE  : {DRIVE_BASE}")
print(f"WHOIS_PATH  : {WHOIS_PATH} ({'encontrado' if os.path.exists(WHOIS_PATH) else 'NÃO encontrado — rode sem WHOIS'})")
print(f"MAX_TRAIN   : {MAX_TRAIN_SAMPLES:,}")
print(f"MAX_EVAL    : {MAX_EVAL_SAMPLES:,}")
print(f"MAX_LENGTH  : {MAX_LENGTH} tokens")
print(f"NUM_EPOCHS  : {NUM_EPOCHS}")

In [ ]:
# ============================================================
# 4. Montar Google Drive e Criar Diretórios
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

for subdir in ['models', 'resultados', 'predicoes', 'plots']:
    os.makedirs(f"{DRIVE_BASE}/{subdir}", exist_ok=True)

print(f"Diretórios criados em: {DRIVE_BASE}")

In [ ]:
# ============================================================
# 5. Carregar Dataset
# ============================================================
print("Carregando dataset kmack/Phishing_urls...")
raw_ds   = load_dataset("kmack/Phishing_urls")
train_df = raw_ds['train'].to_pandas()
test_df  = raw_ds['test'].to_pandas()
valid_df = raw_ds['valid'].to_pandas()
print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | Valid: {len(valid_df):,}")

# Amostragem estratificada
if MAX_TRAIN_SAMPLES and len(train_df) > MAX_TRAIN_SAMPLES:
    frac = MAX_TRAIN_SAMPLES / len(train_df)
    train_df = (
        train_df.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(frac=frac, random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )
    print(f"Amostra treino (estratificada): {len(train_df):,}")

eval_df = pd.concat([test_df, valid_df], ignore_index=True)
if MAX_EVAL_SAMPLES and len(eval_df) > MAX_EVAL_SAMPLES:
    frac = MAX_EVAL_SAMPLES / len(eval_df)
    eval_df = (
        eval_df.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(frac=frac, random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )
    print(f"Amostra avaliação (estratificada): {len(eval_df):,}")

print(f"\nTreino: {len(train_df):,} | Avaliação: {len(eval_df):,}")
print(f"Phishing treino: {train_df['label'].mean():.1%}")

In [ ]:
# ============================================================
# 6. Enriquecimento WHOIS
# ============================================================
def load_whois_cache(path):
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            cache = json.load(f)
        found = sum(1 for v in cache.values() if v.get('status') == 'found')
        print(f"Cache WHOIS: {len(cache):,} domínios ({found:,} com dados, {len(cache)-found:,} not_found)")
        return cache
    print(f"AVISO: '{path}' não encontrado. Rodando sem WHOIS.")
    return {}


def enrich_url(url: str, cache: dict) -> str:
    """
    Combina URL bruta com metadados WHOIS disponíveis.

    Com dados : '[URL] https://... [AGE] 365d [REG] GoDaddy [WHOIS] found'
    Sem dados : '[URL] https://... [WHOIS] unknown'

    Campos incluídos (country omitido — quase sempre 'unknown' no cache):
      - domain_age_days : idade do domínio em dias
      - registrar       : primeiros 25 chars do nome do registrador
      - status          : 'found' ou 'unknown'
    """
    try:
        ext = tldextract.extract(url)
        domain = f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain
    except Exception:
        domain = ""

    info = cache.get(domain, {})
    if info.get('status') == 'found':
        age = info.get('domain_age_days', '?')
        reg = str(info.get('registrar', 'unknown'))[:25].strip()
        return f"[URL] {url} [AGE] {age}d [REG] {reg} [WHOIS] found"
    return f"[URL] {url} [WHOIS] unknown"


whois_cache = load_whois_cache(WHOIS_PATH)

print("Aplicando enriquecimento WHOIS...")
t0 = time.time()
train_texts  = [enrich_url(u, whois_cache) for u in train_df['text']]
eval_texts   = [enrich_url(u, whois_cache) for u in eval_df['text']]
train_labels = train_df['label'].astype(int).tolist()
eval_labels  = eval_df['label'].astype(int).tolist()
print(f"Concluído em {time.time()-t0:.1f}s")

print(f"\nExemplos de entrada:")
for i in [0, 1]:
    print(f"  [{train_labels[i]}] {train_texts[i][:110]}...")

In [ ]:
# ============================================================
# 7. Classes e Funções Compartilhadas
# ============================================================

# --- Dataset PyTorch ---
class URLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


# --- Métricas para o Trainer ---
def compute_hf_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1":       float(f1_score(labels, preds, zero_division=0)),
    }


# --- Métricas completas ---
def compute_full_metrics(y_true, y_pred, y_proba, model_name, train_time, latencies):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Modelo":                  model_name,
        "Accuracy":                float(accuracy_score(y_true, y_pred)),
        "Precision":               float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall":                  float(recall_score(y_true, y_pred, zero_division=0)),
        "F1-Score":                float(f1_score(y_true, y_pred, zero_division=0)),
        "AUC-ROC":                 float(roc_auc_score(y_true, y_proba)),
        "Avg Precision (PR-AUC)": float(average_precision_score(y_true, y_proba)),
        "MCC":                     float(matthews_corrcoef(y_true, y_pred)),
        "Log Loss":                float(log_loss(y_true, y_proba)),
        "Specificity":             float(tn / (tn + fp)),
        "Latência P50 (ms)":       float(np.percentile(latencies, 50)),
        "Latência P95 (ms)":       float(np.percentile(latencies, 95)),
        "Latência P99 (ms)":       float(np.percentile(latencies, 99)),
        "Tempo Treino (s)":        float(train_time),
    }


# --- Função principal: treinar ou carregar do cache, avaliar e salvar ---
def run_model(cfg):
    """
    Executa o pipeline completo para um modelo.

    cfg aceita:
      name        — nome do modelo (usado para cache e CSV)
      model_id    — identificador HuggingFace
      batch_size  — batch size de treino
      grad_accum  — gradient accumulation steps
      max_length  — (opcional) comprimento máximo de sequência; padrão = MAX_LENGTH
                    Para CANINE use 256 (caracteres, não subwords)
      model_class — (opcional) classe explícita do modelo; padrão = AutoModelForSequenceClassification
                    Use CanineForSequenceClassification para CANINE
    """
    name        = cfg["name"]
    model_id    = cfg["model_id"]
    bs          = cfg["batch_size"]
    ga          = cfg["grad_accum"]
    seq_len     = cfg.get("max_length", MAX_LENGTH)
    ModelClass  = cfg.get("model_class", AutoModelForSequenceClassification)

    model_path = f"{DRIVE_BASE}/models/{name}"
    preds_file = f"{DRIVE_BASE}/predicoes/{name}_predictions.npz"

    model_exists = os.path.exists(os.path.join(model_path, "config.json"))
    preds_exist  = os.path.exists(preds_file)

    print(f"\n{'='*65}")
    print(f"Modelo : {name}")
    print(f"ID     : {model_id}")
    print(f"Config : batch={bs} | grad_accum={ga} | efetivo={bs*ga} | max_length={seq_len}")
    print(f"Cache  : modelo={'✓' if model_exists else '✗'} | predições={'✓' if preds_exist else '✗'}")
    print(f"{'='*65}")

    train_time = 0.0

    # ── Passo 1: treinar ou carregar ──────────────────────────────
    if model_exists:
        print("  Carregando modelo do Drive (treino pulado)...")
        tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
        model     = ModelClass.from_pretrained(model_path, num_labels=2)
    else:
        print(f"  Carregando tokenizer e modelo base: {model_id}")
        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
        train_ds  = URLDataset(train_texts, train_labels, tokenizer, seq_len)
        eval_ds   = URLDataset(eval_texts,  eval_labels,  tokenizer, seq_len)

        model = ModelClass.from_pretrained(
            model_id, num_labels=2, ignore_mismatched_sizes=True,
        )
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parâmetros treináveis: {n_params/1e6:.1f}M")

        training_args = TrainingArguments(
            output_dir=f"/content/tmp_{name}",
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=bs,
            per_device_eval_batch_size=bs * 2,
            gradient_accumulation_steps=ga,
            learning_rate=LR,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            logging_steps=500,
            eval_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            fp16=torch.cuda.is_available(),
            bf16=False,
            report_to="none",
            seed=RANDOM_STATE,
            dataloader_num_workers=2,
            dataloader_pin_memory=True,
        )
        trainer = Trainer(
            model=model, args=training_args,
            train_dataset=train_ds, eval_dataset=eval_ds,
            compute_metrics=compute_hf_metrics,
        )

        print(f"  Iniciando treino ({NUM_EPOCHS} épocas)...")
        t_start = time.time()
        trainer.train()
        train_time = time.time() - t_start
        print(f"  Treino concluído: {train_time/60:.1f} min")

        os.makedirs(model_path, exist_ok=True)
        model.save_pretrained(model_path)
        tokenizer.save_pretrained(model_path)
        print(f"  Modelo salvo: {model_path}")

        del trainer, train_ds, eval_ds
        gc.collect()
        torch.cuda.empty_cache()

    # ── Passo 2: predições em batch ────────────────────────────────
    if preds_exist:
        print("  Carregando predições do Drive...")
        data       = np.load(preds_file)
        y_true     = data['y_true']
        y_pred     = data['y_pred']
        y_proba    = data['y_proba']
        train_time = float(data['train_time'][0]) if 'train_time' in data else train_time
    else:
        print("  Gerando predições em batch...")
        model_gpu    = model.to(device)
        model_gpu.eval()
        eval_ds_pred = URLDataset(eval_texts, eval_labels, tokenizer, seq_len)
        loader       = DataLoader(eval_ds_pred, batch_size=bs * 2, shuffle=False, pin_memory=True)

        all_logits = []
        with torch.no_grad():
            for batch in loader:
                inputs  = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
                outputs = model_gpu(**inputs)
                all_logits.append(outputs.logits.cpu().numpy())

        logits  = np.vstack(all_logits)
        y_proba = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
        y_pred  = np.argmax(logits, axis=-1)
        y_true  = np.array(eval_labels)

        np.savez(
            preds_file,
            y_true=y_true, y_pred=y_pred, y_proba=y_proba,
            train_time=np.array([train_time]),
        )
        print(f"  Predições salvas: {preds_file}")

        del eval_ds_pred, loader, all_logits, model_gpu
        gc.collect()
        torch.cuda.empty_cache()

    # ── Passo 3: latência GPU single-sample ────────────────────────
    print("  Medindo latência GPU (500 amostras individuais)...")
    model_gpu = model.to(device)
    model_gpu.eval()
    latencies = []

    for text in eval_texts[:5]:
        enc = tokenizer(
            [text], truncation=True, padding="max_length",
            max_length=seq_len, return_tensors="pt",
        )
        with torch.no_grad():
            _ = model_gpu(**{k: v.to(device) for k, v in enc.items()})

    for text in eval_texts[:500]:
        enc    = tokenizer(
            [text], truncation=True, padding="max_length",
            max_length=seq_len, return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in enc.items()}
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model_gpu(**inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

    latencies = np.array(latencies)

    # ── Passo 4: métricas completas ────────────────────────────────
    row = compute_full_metrics(y_true, y_pred, y_proba, name, train_time, latencies)

    print(f"\n  {'Accuracy':<22}: {row['Accuracy']:.4f}")
    print(f"  {'Precision':<22}: {row['Precision']:.4f}")
    print(f"  {'Recall':<22}: {row['Recall']:.4f}")
    print(f"  {'F1-Score':<22}: {row['F1-Score']:.4f}")
    print(f"  {'AUC-ROC':<22}: {row['AUC-ROC']:.4f}")
    print(f"  {'PR-AUC':<22}: {row['Avg Precision (PR-AUC)']:.4f}")
    print(f"  {'MCC':<22}: {row['MCC']:.4f}")
    print(f"  {'Latência P50':<22}: {row['Latência P50 (ms)']:.2f} ms")
    print(f"  {'Latência P95':<22}: {row['Latência P95 (ms)']:.2f} ms")
    treino_str = f"{train_time/60:.1f} min" if train_time > 0 else "carregado do cache"
    print(f"  {'Tempo treino':<22}: {treino_str}")

    csv_path = f"{DRIVE_BASE}/resultados/resultado_{name}.csv"
    pd.DataFrame([row]).set_index("Modelo").to_csv(csv_path)
    print(f"  Resultado CSV salvo: {csv_path}")

    del model, model_gpu
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"  VRAM livre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    return row, {"y_pred": y_pred, "y_proba": y_proba, "y_true": y_true}


results         = []
all_predictions = {}
print("Classes e funções definidas. Pronto para rodar os modelos.")

---
## Modelos — execute cada célula abaixo individualmente

Cada célula é independente. Se a sessão cair, remonte o Drive (célula 4), recarregue o dataset e WHOIS (células 5–6), redefina as funções (célula 7) e continue de onde parou — os modelos já treinados serão carregados do cache automaticamente.

In [ ]:
# ============================================================
# MODELO 1 — DomURLs-BERT (amahdaouy/DomURLs_BERT)
# ~110M parâmetros | BERT-base pré-treinado em URLs e domínios
# maliciosos (DGA, DNS tunneling, phishing) — arXiv:2409.09143
# Vocabulário customizado de 32k tokens focado em URLs
# Tempo estimado T4: ~30–45 min (200k amostras, 3 épocas)
# ============================================================
_row, _preds = run_model({
    "name":       "DomURLs-BERT",
    "model_id":   "amahdaouy/DomURLs_BERT",
    "batch_size": 64,
    "grad_accum": 1,
})
results.append(_row)
all_predictions["DomURLs-BERT"] = _preds

In [ ]:
# ============================================================
# MODELO 2 — CANINE-s (google/canine-s)
# ~121M parâmetros | Transformer de nível de caractere
# Sem tokenização por subpalavras — opera em Unicode code points
# Arquitetura: local attention + downsampling/upsampling
# max_length=256 caracteres (≈ URLs de até 256 chars, cobre >99%)
# Tempo estimado T4: ~40–55 min (200k amostras, 3 épocas)
# ============================================================
_row, _preds = run_model({
    "name":        "CANINE-s",
    "model_id":    "google/canine-s",
    "batch_size":  32,
    "grad_accum":  2,
    "max_length":  256,                        # caracteres, não subwords
    "model_class": CanineForSequenceClassification,  # Auto não resolve CANINE
})
results.append(_row)
all_predictions["CANINE-s"] = _preds

In [ ]:
# ============================================================
# MODELO 3 — XLNet-base (xlnet-base-cased)
# ~117M parâmetros | Permutation Language Modeling
# Autoregressive bidirecional — fundamentalmente diferente do
# masked LM do BERT. Usa Transformer-XL com atenção relativa.
# Tempo estimado T4: ~45–60 min (200k amostras, 3 épocas)
# ============================================================
_row, _preds = run_model({
    "name":       "XLNet-base",
    "model_id":   "xlnet-base-cased",
    "batch_size": 32,
    "grad_accum": 2,
})
results.append(_row)
all_predictions["XLNet-base"] = _preds

In [ ]:
# ============================================================
# MODELO 4 — SecureBERT (ehsanaghaei/SecureBERT)
# ~125M parâmetros | RoBERTa-base pré-treinado em corpus de
# cibersegurança (CVEs, threat reports, security blogs)
# Complementa DomURLs-BERT: domínio cyber amplo vs URLs específicas
# Tempo estimado T4: ~35–50 min (200k amostras, 3 épocas)
# ============================================================
_row, _preds = run_model({
    "name":       "SecureBERT",
    "model_id":   "ehsanaghaei/SecureBERT",
    "batch_size": 64,
    "grad_accum": 1,
})
results.append(_row)
all_predictions["SecureBERT"] = _preds

---
## Análise e Visualizações

Execute as células abaixo após concluir todos os modelos desejados.

In [ ]:
# ============================================================
# 8. Tabela Comparativa
# ============================================================
df_results = pd.DataFrame(results).set_index("Modelo")

def highlight_best(s):
    if s.name == "Log Loss" or "Latência" in s.name or "Tempo" in s.name:
        is_best = s == s.min()
    else:
        is_best = s == s.max()
    return ["background-color: #c6efce; font-weight: bold" if v else "" for v in is_best]

styled = (
    df_results.style
    .apply(highlight_best, axis=0)
    .format({
        "Accuracy":                "{:.4f}",
        "Precision":               "{:.4f}",
        "Recall":                  "{:.4f}",
        "F1-Score":                "{:.4f}",
        "AUC-ROC":                 "{:.4f}",
        "Avg Precision (PR-AUC)": "{:.4f}",
        "MCC":                     "{:.4f}",
        "Log Loss":                "{:.4f}",
        "Specificity":             "{:.4f}",
        "Latência P50 (ms)":       "{:.2f}",
        "Latência P95 (ms)":       "{:.2f}",
        "Latência P99 (ms)":       "{:.2f}",
        "Tempo Treino (s)":        "{:.0f}",
    })
    .set_caption("Benchmark — Modelos Grandes (Transformers) | kmack/Phishing_urls + WHOIS")
)

display(styled)
print("\n" + "="*80)
print(df_results.round(4).to_string())

In [ ]:
# ============================================================
# 9. Matrizes de Confusão
# ============================================================
n = len(all_predictions)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5))
if n == 1: axes = [axes]

for ax, (name, preds) in zip(axes, all_predictions.items()):
    cm = confusion_matrix(preds["y_true"], preds["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Legítimo", "Phishing"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d"
    )
    ax.set_title(name, fontsize=12, fontweight="bold")

plt.suptitle("Matrizes de Confusão — Modelos Grandes", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
path = f"{DRIVE_BASE}/plots/matrizes_confusao_grandes.png"
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvo: {path}")

In [ ]:
# ============================================================
# 10. Curvas ROC e Precision-Recall
# ============================================================
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

ax = axes[0]
for (name, preds), color in zip(all_predictions.items(), colors):
    fpr, tpr, _ = roc_curve(preds["y_true"], preds["y_proba"])
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc(fpr,tpr):.4f})")
ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.5,label="Random")
ax.set_xlabel("FPR", fontsize=12); ax.set_ylabel("TPR", fontsize=12)
ax.set_title("Curvas ROC — Modelos Grandes", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=9); ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01,1.01]); ax.set_ylim([-0.01,1.01])

ax = axes[1]
for (name, preds), color in zip(all_predictions.items(), colors):
    p, r, _ = precision_recall_curve(preds["y_true"], preds["y_proba"])
    ap = average_precision_score(preds["y_true"], preds["y_proba"])
    ax.plot(r, p, color=color, lw=2, label=f"{name} (AP={ap:.4f})")
ax.set_xlabel("Recall", fontsize=12); ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Curvas Precision-Recall — Modelos Grandes", fontsize=14, fontweight="bold")
ax.legend(loc="lower left", fontsize=9); ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01,1.01]); ax.set_ylim([-0.01,1.01])

plt.tight_layout()
path = f"{DRIVE_BASE}/plots/curvas_roc_pr_grandes.png"
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show(); print(f"Salvo: {path}")

In [ ]:
# ============================================================
# 11. Curvas ROC Individuais
# ============================================================
n = len(all_predictions)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4.5))
if n == 1: axes = [axes]

for ax, ((name, preds), color) in zip(axes, zip(all_predictions.items(), colors)):
    fpr, tpr, _ = roc_curve(preds["y_true"], preds["y_proba"])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2)
    ax.fill_between(fpr, tpr, alpha=0.15, color=color)
    ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.4)
    ax.set_title(f"{name}\nAUC = {roc_auc:.4f}", fontsize=12, fontweight="bold")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.grid(True, alpha=0.3)

plt.suptitle("Curvas ROC Individuais — Modelos Grandes", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
path = f"{DRIVE_BASE}/plots/curvas_roc_individuais_grandes.png"
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show(); print(f"Salvo: {path}")

In [ ]:
# ============================================================
# 12. Comparação de Métricas (Barras)
# ============================================================
df_plot = df_results[["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC", "MCC"]]
fig, ax = plt.subplots(figsize=(16, 6))
df_plot.plot(kind="bar", ax=ax, width=0.75, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Comparação de Métricas — Modelos Grandes (Transformers)", fontsize=14, fontweight="bold")
ax.set_xticklabels(df_plot.index, rotation=10, ha="right", fontsize=10)
ax.set_ylim(0.70, 1.03)
ax.legend(loc="lower right", fontsize=9, ncol=2)
ax.grid(axis="y", alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=7, padding=2)
plt.tight_layout()
path = f"{DRIVE_BASE}/plots/comparacao_metricas_grandes.png"
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show(); print(f"Salvo: {path}")

In [ ]:
# ============================================================
# 13. Comparação de Latência GPU
# ============================================================
df_lat = df_results[["Latência P50 (ms)", "Latência P95 (ms)", "Latência P99 (ms)"]]
fig, ax = plt.subplots(figsize=(12, 5))
df_lat.plot(kind="bar", ax=ax, width=0.7, edgecolor="black", linewidth=0.5,
            color=["#66b3ff", "#ff9999", "#ff6666"])
ax.set_ylabel("Latência (ms)", fontsize=12)
ax.set_title("Latência GPU de Inferência (single-sample)", fontsize=14, fontweight="bold")
ax.set_xticklabels(df_lat.index, rotation=10, ha="right", fontsize=10)
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=8, padding=2)
plt.tight_layout()
path = f"{DRIVE_BASE}/plots/comparacao_latencia_grandes.png"
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show(); print(f"Salvo: {path}")

In [ ]:
# ============================================================
# 14. Comparação com Modelos Leves (se disponível)
# ============================================================
# Coloque 'resultados_modelos_leves.csv' no Drive ou em /content/
LEVES_CSV = f"{DRIVE_BASE}/resultados/resultados_modelos_leves.csv"
if not os.path.exists(LEVES_CSV):
    LEVES_CSV = "/content/resultados_modelos_leves.csv"

try:
    df_leves = pd.read_csv(LEVES_CSV, index_col="Modelo")
    df_comb  = pd.concat([df_leves, df_results], axis=0)

    cores = (["#aaaaaa"] * len(df_leves)) + colors[:len(df_results)]
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    for ax, metric in zip(axes, ["F1-Score", "AUC-ROC"]):
        bars = ax.bar(df_comb.index, df_comb[metric], color=cores, edgecolor="black", linewidth=0.5)
        ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)
        ax.set_ylabel(metric, fontsize=12)
        ax.set_title(f"{metric}: Modelos Leves vs Grandes", fontsize=13, fontweight="bold")
        ax.set_xticklabels(df_comb.index, rotation=20, ha="right", fontsize=9)
        ax.set_ylim(0.65, 1.03); ax.grid(axis="y", alpha=0.3)
        ax.axvline(x=len(df_leves) - 0.5, color="red", linestyle="--", alpha=0.6, label="Leves | Grandes")
        ax.legend(fontsize=9)

    plt.suptitle("Comparação Geral: Modelos Leves vs Modelos Grandes",
                 fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    path = f"{DRIVE_BASE}/plots/comparacao_leves_vs_grandes.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show(); print(f"Salvo: {path}")

except FileNotFoundError:
    print(f"'{LEVES_CSV}' não encontrado. Copie o CSV dos modelos leves para gerar este gráfico.")

In [ ]:
# ============================================================
# 15. Exportar Tabela Final no Drive
# ============================================================
csv_final = f"{DRIVE_BASE}/resultados/resultados_modelos_grandes.csv"
df_results.to_csv(csv_final)
print(f"Tabela final salva: {csv_final}")

print("\n" + "="*80)
print("RESUMO FINAL")
print("="*80)
cols = ["F1-Score", "AUC-ROC", "MCC", "Latência P50 (ms)", "Tempo Treino (s)"]
print(df_results[cols].round(4).to_string())